# [M7] DỰ ÁN AI THẬT ĐỂ ĐƯA VÀO PORTFOLIO
## Buổi S04: Đánh Giá Chi Tiết, Phân Tích Lỗi & Interactive Demo

---

### Metadata Khóa Học
* **Module:** M7 — DỰ ÁN AI THẬT ĐỂ ĐƯA VÀO PORTFOLIO
* **Buổi:** S04
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook phân tích lỗi chi tiết và ứng dụng Interactive AI Demo
* **Mức độ liên quan VOAI / Olympic AI:** Biến thuật toán thành sản phẩm thực tế có khả năng trình diễn trực tiếp trước giám khảo.

#### Mục tiêu học tập (Learning Objectives)
- Tiến hành Phân tích Lỗi sai Chuyên sâu (Granular Error Analysis): Bóc tách các ca mô hình tự tin sai lầm nghiêm trọng (High-Confidence Failures).
- Đánh giá hiệu năng đa chiều theo từng phân nhóm đối tượng (Slice-based Performance Evaluation) để phát hiện rủi ro thiên vị.
- Xây dựng một Ứng dụng Demo Tương tác Trực tiếp (Interactive Prediction Demo) gọn nhẹ, trực quan và chạy độc lập.
- Thiết kế cơ chế giải thích quyết định dự đoán (Prediction Explainability) kèm thanh phần trăm độ tự tin cho người dùng cuối.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M7_S03 (Đã có mô hình tối ưu từ Ablation Study).
- Nắm vững các metric đánh giá chi tiết từ Module M2 và M4.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Một mô hình AI dù có đạt độ chính xác 95% trên tập dữ liệu kiểm tra thì nó vẫn hoàn toàn "vô hình" và vô giá trị đối với người dùng cuối, nhà đầu tư hoặc ban giám khảo nếu nó chỉ nằm im lìm dưới dạng các dòng code trong một cuốn notebook. Người ta không thể hình dung mô hình của bạn hoạt động thế nào trong đời sống thực!

Hơn thế nữa, 5% sai số còn lại là những trường hợp nào? Mô hình có xu hướng đoán sai ở người cao tuổi hay người trẻ tuổi? Buổi học hôm nay giải quyết trọn vẹn hai câu hỏi sống còn đó: **Mổ xẻ tận gốc các ca dự đoán sai lầm (Error Analysis)** và **Thổi hồn vào mô hình bằng một Ứng dụng Demo Tương tác Trực tiếp (Interactive Demo)** — biến thuật toán phức tạp thành một sản phẩm công nghệ sống động mà bất kỳ ai cũng có thể tự tay thử nghiệm và trầm trồ!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Phương pháp Phân tích Hiệu năng theo Lát cắt (Slice-based Evaluation)
* Không chỉ nhìn vào con số trung bình toàn cục! Một mô hình có Accuracy 90% toàn diện nhưng có thể chỉ đạt 45% trên nhóm người cao tuổi ($Age \ge 65$) $\implies$ Rủi ro y tế chết người!
* **Cách thực hiện:** Chia nhỏ tập kiểm tra thành các lát cắt nhân khẩu học/đặc trưng (Sub-cohort slices) và đo lường metric độc lập trên từng lát cắt để phát hiện vùng trũng hiệu năng.

### 2. Kiến trúc Ứng dụng Demo Tương tác (Interactive Inference Engine)
Một ứng dụng demo đạt chuẩn phải hội tụ đủ 3 yếu tố:
1. **User Input Interface:** Cho phép người dùng nhập thông số tùy ý (hoặc chọn các mẫu thử có sẵn).
2. **End-to-End Inference Pipeline:** Tự động tiền xử lý dữ liệu nhập vào qua Pipeline đã huấn luyện từ Buổi 2.
3. **Visual Output with Confidence & Explanation:** Xuất kết quả kèm theo thanh phần trăm xác suất tự tin ($P \in [0\%, 100\%]$) và cảnh báo trực quan (Xanh / Đỏ).



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Kiến trúc Ứng dụng Demo Tương tác (Interactive Demo Engine):
[GIAO DIỆN NGƯỜI DÙNG: Nhập Tuổi, Huyết áp, Cholesterol, Tình trạng Hút thuốc]
                               │
                               ▼
            [Tự động gọi: preprocessor.transform()]
                               │
                               ▼
                 [Mô hình Tối ưu: model.predict_proba()]
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│ KẾT QUẢ DỰ ĐOÁN: BỆNH NHÂN CÓ NGUY CƠ TIM MẠCH CAO (NGUY HIỂM!)            │
│ Độ tự tin của mô hình: [████████████████████░░] 87.5%                       │
│ Yếu tố cảnh báo chính: Huyết áp vượt ngưỡng (165 mmHg) & Tuổi cao (68 tuổi) │
└─────────────────────────────────────────────────────────────────────────────┘
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report

# 1. PHAN TICH HIEU NANG THEO LAT CAT (SLICE-BASED ERROR ANALYSIS)
np.random.seed(42)
n_test = 150

# Gia lap tap test gom Tuoi, Huyet ap va Nhan thuc te
test_age = np.random.randint(20, 80, size=n_test)
test_bp = np.random.normal(130, 20, size=n_test).round(1)
true_target = (test_age * 0.04 + test_bp * 0.03 + np.random.randn(n_test) > 5.5).astype(int)

# Gia lap du doan cua mo hinh
pred_probs = 1 / (1 + np.exp(-(test_age * 0.04 + test_bp * 0.03 - 5.3)))
pred_target = (pred_probs >= 0.5).astype(int)

df_analysis = pd.DataFrame({
    'Age': test_age,
    'Blood_Pressure': test_bp,
    'True_Label': true_target,
    'Pred_Label': pred_target,
    'Confidence': np.maximum(pred_probs, 1 - pred_probs)
})

# Phan chia thanh 2 lat cat: Nguoi tre (< 50 tuoi) vs Nguoi cao tuoi (>= 50 tuoi)
young_mask = df_analysis['Age'] < 50
senior_mask = df_analysis['Age'] >= 50

acc_young = (df_analysis[young_mask]['True_Label'] == df_analysis[young_mask]['Pred_Label']).mean()
acc_senior = (df_analysis[senior_mask]['True_Label'] == df_analysis[senior_mask]['Pred_Label']).mean()

print("===== PHAN TICH HIEU NANG THEO LAT CAT NHAN KHAU HOC (SLICING) =====")
print(f"Tong so mau Test: {n_test} mau")
print(f"Do chinh xac nhom Nguoi Tre (< 50 tuoi)      : {acc_young*100:.1f}% ({young_mask.sum()} mau)")
print(f"Do chinh xac nhom Nguoi Cao Tuoi (>= 50 tuoi): {acc_senior*100:.1f}% ({senior_mask.sum()} mau)")
print("-> PHAT HIEN: Mo hinh co su chenh lech hieu nang giua cac nhom tuoi. Can them du lieu cho nhom yeu hon!")

### 🔍 Code Walkthrough (Khối 1)
Khối code thực thi kỹ thuật phân tích hiệu năng theo lát cắt (Slice-based Evaluation):
- Chia nhỏ tập dữ liệu kiểm thử theo độ tuổi để phát hiện điểm mù của mô hình.
- Cung cấp bằng chứng định lượng rõ ràng giúp thí sinh đưa ra các giải pháp cải tiến chính xác.


In [ ]:
# 2. XAY DUNG UNG DUNG DEMO TUONG TAC TRUC TIEP (INTERACTIVE PREDICTOR APP)
# Day la ma nguon san sang tich hop vao CLI hoac giao dien Web (Streamlit/Gradio)

def run_interactive_ai_demo(patient_age, blood_pressure, cholesterol, smoking_status):
    print("\n==================================================================")
    print("      HE THONG AI HO TRO CHUAN DOAN NGUY CO TIM MACH (DEMO)      ")
    print("==================================================================")
    print(f"Thong tin dau vao: Tuoi = {patient_age} | Huyet ap = {blood_pressure} mmHg | Mo mau = {cholesterol} mg/dL | Hut thuoc = {smoking_status}")
    
    # Mo phong pipeline tien xu ly va suy dien (Inference)
    # Logits gia lap tinh toan dua tren cac trong so da hoc
    smoke_score = 1.2 if smoking_status == 'Current' else (0.5 if smoking_status == 'Former' else 0.0)
    risk_score = (patient_age - 45) * 0.05 + (blood_pressure - 120) * 0.04 + (cholesterol - 180) * 0.02 + smoke_score
    prob_risk = 1 / (1 + np.exp(-risk_score))
    
    # Dinh dang thanh tien do truc quan
    bar_length = 20
    filled_length = int(bar_length * prob_risk)
    bar = "█" * filled_length + "░" * (bar_length - filled_length)
    
    print("\n--- KET QUA PHAN TICH TU MO HINH AI ---")
    print(f"Do tin cay nguy co: [{bar}] {prob_risk*100:.1f}%")
    
    if prob_risk >= 0.65:
        print("🚨 KET LUAN: NGUY CO CAO (HIGH RISK) - Can can thiep y te som!")
        print("💡 Loi khuyen AI: Kiem soat huyet ap va ngung hut thuoc la ngay lap tuc.")
    elif prob_risk >= 0.35:
        print("⚠️ KET LUAN: NGUY CO TRUNG BINH (MODERATE RISK) - Can theo doi them.")
        print("💡 Loi khuyen AI: Cai thien che do an uong va tap the duc thuong xuyen.")
    else:
        print("✅ KET LUAN: NGUY CO THAP (LOW RISK) - Chi so tim mach on dinh.")
        print("💡 Loi khuyen AI: Duy tri loi song lanh manh hien tai.")
    print("==================================================================")

# Chay thu nghiem Demo tren 2 ca benh nhan thuc te
run_interactive_ai_demo(patient_age=65, blood_pressure=155.0, cholesterol=240.0, smoking_status='Current')
run_interactive_ai_demo(patient_age=28, blood_pressure=115.0, cholesterol=165.0, smoking_status='Never')

### 🔍 Code Walkthrough (Khối 2)
Khối code lập trình một ứng dụng Demo tương tác hoàn chỉnh:
- Nhận thông số đầu vào linh hoạt của bất kỳ người dùng nào.
- Trực quan hóa phần trăm độ tự tin bằng thanh tiến độ (Visual ASCII Bar).
- Xuất kết luận phân loại rõ ràng kèm lời khuyên hành động thực tế, sẵn sàng trình diễn trực tiếp trước hội đồng giám khảo.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Đóng gói Demo thành file `app.py` độc lập
Trích xuất đoạn mã hàm `run_interactive_ai_demo` và lưu thành file `portfolio_project/app.py` độc lập. Chạy thử từ dòng lệnh terminal bằng lệnh `python portfolio_project/app.py` để xác minh chương trình chạy trơn tru.

### Bài 2 (Easy): Mở rộng giao diện nhập liệu từ bàn phím
Sửa đổi hàm demo để sử dụng hàm `input()` của Python cho phép người dùng tự gõ các chỉ số từ bàn phím tại thời điểm runtime thay vì gán cứng tham số.

### Bài 3 (Medium): Trích xuất 5 ca dự đoán sai tồi tệ nhất (Top-5 Worst Failures)
Viết hàm lọc ra 5 trường hợp trong bảng `df_analysis` mà mô hình có độ tự tin cao nhất ($P > 90\%$) nhưng lại dự đoán sai hoàn toàn nhãn thực tế. Phân tích đặc điểm chung của 5 trường hợp này.

### Bài 4 (Medium): Đánh giá tính giải thích được với SHAP / LIME Intuition
Tìm hiểu khái niệm SHAP (SHapley Additive exPlanations). Viết một đoạn văn ngắn giải thích làm thế nào SHAP có thể bóc tách đóng góp của từng đặc trưng (ví dụ: Huyết áp đóng góp $+35\%$ vào nguy cơ bệnh) cho từng bệnh nhân cụ thể.

### Bài 5 (Challenge): Xây dựng Web Demo bằng Streamlit hoặc Gradio
Tìm hiểu thư viện Gradio hoặc Streamlit. Viết một file `web_demo.py` ngắn dưới 30 dòng code tạo ra một trang web tương tác cục bộ có các thanh trượt (Sliders) và nút bấm (Button) để dự đoán trực quan trên trình duyệt.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong phần thi chung kết Olympic AI hoặc triển lãm sản phẩm nghiên cứu, **một ứng dụng Demo chạy mượt mà trước mắt Ban giám khảo có giá trị thuyết phục gấp 100 lần những lời nói lý thuyết suông**. Giám khảo luôn yêu cầu: *"Hãy nhập thử một dữ liệu bất kỳ do tôi đưa ra xem mô hình của bạn phản hồi thế nào?"*. Chuẩn bị sẵn một Demo Engine tương tác hoàn hảo giúp bạn làm chủ hoàn toàn sân khấu và nắm chắc điểm số tối đa.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Hàm demo nhận dữ liệu đầu vào từ người dùng nhưng bị lỗi sập chương trình khi người dùng nhập sai kiểu dữ liệu (ví dụ gõ chữ "sáu mươi" vào ô tuổi, hoặc nhập số âm):

```python
# CODE CÓ LỖI THIẾU KIỂM TRA ĐẦU VÀO NGƯỜI DÙNG:
# age = int(user_input) # Neu user go "abc" -> ValueError: invalid literal for int()!
```

**Nhiệm vụ của bạn:**
1. Viết khối lệnh bọc an toàn `try-except` và kiểm tra miền giá trị hợp lệ trước khi đưa vào mô hình.
2. Trả về thông báo hướng dẫn thân thiện: *"Vui lòng nhập tuổi là số nguyên từ 1 đến 120"*.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Hoạt động "Slice-based Performance Evaluation" (Đánh giá hiệu năng theo lát cắt) có ý nghĩa gì đối với một dự án AI chuẩn mực?
   - A. Cắt nhỏ file code thành nhiều phần.
   - B. Đánh giá chất lượng mô hình trên từng phân nhóm dữ liệu cụ thể (như độ tuổi, giới tính, vùng miền) để phát hiện các điểm mù và bảo đảm tính công bằng, an toàn trước khi triển khai.
   - C. Giảm độ phân giải của hình ảnh.
   - D. Tăng tốc độ quạt chip CPU.

2. Tại sao một dự án AI hoàn chỉnh đưa vào Portfolio bắt buộc phải có một sản phẩm Demo tương tác (Interactive Demo)?
   - A. Để làm cho bài tập dài hơn.
   - B. Biến mô hình toán học trừu tượng thành một sản phẩm công nghệ trực quan, cho phép người dùng và nhà tuyển dụng trực tiếp thử nghiệm và kiểm chứng năng lực giải quyết vấn đề thực tế.
   - C. Vì luật thi bắt buộc phải dùng web.
   - D. Để tải thêm nhiều file từ Internet.

3. Trong ứng dụng demo dự đoán, thông tin "Độ tự tin" (Confidence Score) mang lại giá trị gì cho người dùng cuối?
   - A. Cho biết tốc độ đường truyền mạng.
   - B. Cung cấp mức độ chắc chắn của mô hình về quyết định đưa ra (ví dụ 95% chắc chắn vs 51% phân vân), giúp con người biết khi nào nên tin tưởng và khi nào cần kiểm tra lại cẩn trọng.
   - C. Cho biết dung lượng pin còn lại.
   - D. Đo lường nhiệt độ máy chủ.

4. Trường hợp nào sau đây được coi là lỗi thất bại nghiêm trọng nhất của mô hình trong quá trình phân tích lỗi sai (Error Analysis)?
   - A. Mô hình đoán sai nhưng độ tự tin thấp ($P pprox 50\%$).
   - B. Mô hình đoán sai hoàn toàn nhưng lại có độ tự tin cực cao ($P > 95\%$ - High-Confidence Failure).
   - C. Mô hình chạy mất 2 giây.
   - D. Mô hình in ra kết quả màu xanh.

5. Sản phẩm mà học viên xây dựng và kiểm thử thành công trong Buổi M7_S04 là gì?
   - A. Một bài báo in trên giấy.
   - B. Báo cáo phân tích lỗi sai chuyên sâu (Error Analysis) kèm Ứng dụng Demo Tương tác Trực tiếp (Interactive Prediction Demo).
   - C. Một chứng chỉ giả định.
   - D. Một bài hát về AI.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Kỹ thuật đánh giá hiệu năng theo lát cắt phân nhóm (Slice-based Evaluation).
  - Phương pháp mổ xẻ các ca thất bại có độ tự tin cao (High-Confidence Failures).
  - Kiến trúc xây dựng Ứng dụng Demo Tương tác Trực tiếp (Interactive Demo Engine).
* **What I can now do:**
  - Biến mô hình AI tĩnh trong notebook thành một ứng dụng tương tác sống động có thanh đo độ tự tin.
  - Tự tin phân tích điểm yếu và ranh giới an toàn của mô hình trước người dùng.
* **What I still need to practice:**
  - Đóng gói toàn bộ dự án vào **GitHub Repo, Báo cáo Kỹ thuật & Slide Thuyết trình** ở Buổi 5!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Buổi tiếp theo (**M7_S05**) là buổi tổng kết đỉnh cao và là đích đến cuối cùng của toàn bộ khóa học 38 buổi: **Hoàn Thiện Portfolio: Đóng Gói GitHub Repo, Viết Báo Cáo Kỹ Thuật & Thuyết Trình**! Bạn sẽ hoàn thiện trang README đẳng cấp, đóng gói mã nguồn, diễn tập thuyết trình và chính thức nhận Chứng chỉ tốt nghiệp toàn khóa **Final Project Certificate**!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M7**:
- Đánh giá hiệu năng theo lát cắt (Slice-based evaluation) ✓
- Phân tích lỗi sai chi tiết (High-confidence error analysis) ✓
- Xây dựng Interactive Prediction Demo có đo lường độ tự tin ✓

* **Session Deliverable:** Mã nguồn Interactive Prediction Demo hoàn chỉnh kèm phân tích lỗi
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
